# ETL Extract Phase — COVID-19 Data    

This notebook handles the **Extract** phase of the ETL process.  
It loads, inspects, and validates the COVID-19 dataset obtained from Kaggle  
([Our World in Data](https://www.kaggle.com/datasets/georgesaavedra/covid19-dataset)).


In [2]:
import os

# Change directory to your project folder
os.chdir(r"C:\Users\USER\Desktop\Data mining and warehousing\ET_Exam_Tumaini_933")

# Confirm current working directory
print("Now working in:", os.getcwd())


Now working in: C:\Users\USER\Desktop\Data mining and warehousing\ET_Exam_Tumaini_933


In [3]:
import pandas as pd

# Load both datasets from the data folder
raw_df = pd.read_csv('data/raw_data.csv')
incremental_df = pd.read_csv('data/incremental_data.csv')

print("Raw dataset shape:", raw_df.shape)
print("Incremental dataset shape:", incremental_df.shape)

# Preview the first few rows
raw_df.head()


Raw dataset shape: (166326, 67)
Incremental dataset shape: (1140, 67)


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN


In [4]:
import os
print(os.getcwd())


C:\Users\USER\Desktop\Data mining and warehousing\ET_Exam_Tumaini_933


In [5]:
raw_df.info()
raw_df.describe(include='all').T.head(10)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166326 entries, 0 to 166325
Data columns (total 67 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   iso_code                                    166326 non-null  object 
 1   continent                                   156370 non-null  object 
 2   location                                    166326 non-null  object 
 3   date                                        166326 non-null  object 
 4   total_cases                                 163293 non-null  float64
 5   new_cases                                   163133 non-null  float64
 6   new_cases_smoothed                          161150 non-null  float64
 7   total_deaths                                145451 non-null  float64
 8   new_deaths                                  145487 non-null  float64
 9   new_deaths_smoothed                         143390 non-null  float64
 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
iso_code,166326,238,ARG,795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
continent,156370,6,Africa,39417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,166326,238,Argentina,795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,166326,795,2021-08-28,238,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_cases,163293.0,NaN,NaN,NaN,2536044.011831,15434413.355228,1.0,2001.0,26117.0,298702.0,445129499.0
new_cases,163133.0,NaN,NaN,NaN,11570.839634,84425.980167,0.0,1.0,79.0,1063.0,4206334.0
new_cases_smoothed,161150.0,NaN,NaN,NaN,11565.595138,82578.300699,0.0,7.0,107.143,1146.0,3444236.714
total_deaths,145451.0,NaN,NaN,NaN,57664.07353,302114.502476,1.0,79.0,783.0,7307.0,5995245.0
new_deaths,145487.0,NaN,NaN,NaN,171.137304,832.251328,0.0,0.0,2.0,20.0,18020.0
new_deaths_smoothed,143390.0,NaN,NaN,NaN,172.673031,817.024076,0.0,0.143,2.429,21.286,14689.143


## Data Quality Issues
1. **Missing values** — Many columns (e.g., ICU data, vaccinations) have NaNs.  
2. **Mixed data types** — The `date` column is stored as a string.  
3. **Incomplete continent data** — Some rows like “World” or “Africa” have `continent = NaN`.  
4. **Potential duplicates** — Check if `(location, date)` repeats.


In [6]:
duplicate_count = raw_df.duplicated(subset=['location', 'date']).sum()
missing_summary = raw_df.isna().sum().sort_values(ascending=False).head(10)

print("Duplicate (location, date) pairs:", duplicate_count)
print("\nTop 10 columns with missing values:")
print(missing_summary)


Duplicate (location, date) pairs: 0

Top 10 columns with missing values:
weekly_icu_admissions_per_million          160893
weekly_icu_admissions                      160893
excess_mortality_cumulative_per_million    160630
excess_mortality                           160630
excess_mortality_cumulative                160630
excess_mortality_cumulative_absolute       160630
weekly_hosp_admissions_per_million         155403
weekly_hosp_admissions                     155403
total_boosters                             148787
total_boosters_per_hundred                 148787
dtype: int64


In [7]:
raw_df['date'] = pd.to_datetime(raw_df['date'], errors='coerce')
incremental_df['date'] = pd.to_datetime(incremental_df['date'], errors='coerce')

merged_df = pd.concat([raw_df, incremental_df]).drop_duplicates(subset=['location', 'date'])
print("Merged dataset shape:", merged_df.shape)


Merged dataset shape: (166326, 67)


In [8]:
merged_df.to_csv('data/validated_data.csv', index=False)
print("Validated data saved to data/validated_data.csv")


Validated data saved to data/validated_data.csv


After concatenating the raw and incremental datasets, duplicate records based on `(location, date)` were dropped to ensure each location-date pair appears only once in the final dataset.


In [9]:
merged_df.head()


,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,female_smokers,male_smokers,handwashing_facilities,hospital_beds_per_thousand,life_expectancy,human_development_index,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million
0,AFG,Asia,Afghanistan,2020-02-24,5.0,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-02-25,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-02-26,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-02-27,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-02-28,5.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,37.746,0.5,64.83,0.511,NaN,NaN,NaN,NaN
